In [ ]:
import pandas as pd
import re
# Load data and data dictionary
data = pd.read_excel("va_data.xlsx", sheet_name="Sheet1")
dictionary = pd.read_excel("data_dictionary.xlsx", sheet_name="Sheet1")

In [ ]:
dictionary[0:10]

,variable,variable_desc,data_type,value_labels,min_value,max_value
0,ident,Main unique identifier in Karonga,text,NaN,NaN,NaN
1,sex,Sex,categorical,"Female=0, Male = 1",NaN,NaN
2,age,Age,numeric,NaN,NaN,NaN
3,vtelldied,Hospital told COD,categorical,"No=0, Yes=1, Refused to answer=6, Not applicab...",NaN,NaN
4,vtelldiedtx,What hosp said was COD,text,NaN,NaN,NaN
5,fullshortva,Do you wish to carry out the full or short VA?,categorical,"Short=1, Long=2",NaN,NaN
6,vknowcod,Know COD,categorical,"No=0, Yes=1, Unknown=8, Missing=9",NaN,NaN
7,vknowcodtx,Respondent COD,text,NaN,NaN,NaN
8,vtb,TB,categorical,"No=0, Yes=1, Unknown=8, Missing=9",NaN,NaN
9,vtbtrtwhere,Where TB treated,text,NaN,NaN,NaN


In [ ]:
data[0:5]

,ident,sex,age,vtelldied,vtelldiedtx,fullshortva,vknowcod,vknowcodtx,vtb,vtbtrtwhere,...,ccod2,cintent,ccomments,consensus,summarydate,summaryby,oldsystem,revsys,recode2014,recode2014reas
0,21259926,NaN,NaN,0.0,NaN,NaN,0.0,NaN,0.0,NaN,...,NaN,NaN,NaN,2.0,2007-05-07,FL,NaN,2.0,NaN,NaN
1,21263273,1.0,49.500000,NaN,NaN,NaN,1.0,CERVICAL CANCER,0.0,NaN,...,142.0,NaN,On ART,3.0,2015-07-14,LH,NaN,2.0,NaN,NaN
2,21264664,0.0,69.500000,1.0,STROKE,NaN,1.0,STROKE,0.0,NaN,...,NaN,NaN,NaN,3.0,2016-04-27,JP,NaN,2.0,NaN,NaN
3,21265586,1.0,58.599998,1.0,HEART VALVE DISEASE,NaN,1.0,HEART VALVE DISEASE,0.0,NaN,...,NaN,NaN,NaN,3.0,2018-09-11,JP,NaN,2.0,NaN,NaN
4,21266393,0.0,83.400002,0.0,NaN,NaN,9.0,NaN,0.0,NaN,...,NaN,NaN,NaN,2.0,2009-06-16,U3,NaN,2.0,NaN,NaN


In [ ]:
def create_demographic_intro(row):
    parts = []
    age = row.get('age')
    sex = row.get('sex')
    sex_text = None
    # Decode sex
    try:
      sex_str = str(int(float(sex)))
    except:
      sex_str = str(sex).strip().lower()

    sex_map = {
        "1": "female",
        "0": "male"
    }

    sex_text = sex_map.get(sex_str, sex_str)
    # Build sentence
    if not pd.isna(age) and sex_text:
        return f"The deceased was a {int(age)}-year-old {sex_text}."
    elif not pd.isna(age):
        return f"The deceased was {int(age)} years old."
    elif sex_text:
        return f"The deceased was {sex_text}."
    return ""

In [ ]:
# Convert coded values like "No=0, Yes=1, Unknown=8"
def parse_codes(code_string):
    if pd.isna(code_string):
        return {}
    pairs = re.findall(r'([^=,]+)=([^,]+)', str(code_string))
    return {v.strip(): k.strip() for k, v in pairs}

def clean_label(label):
    # remove numeric tags like "-2000 fever"
    label = re.sub(r"-?\d+\s*\w*$", "", label)
    return label.strip()

def describe_value(var, value):
    # Skip truly missing values
    if pd.isna(value):
        return None

    # Convert to string for checking
    value_str = str(value).strip().lower()

    # Skip blank or textual missing values
    if value_str in ["", "na", "nan", "none"]:
        return None

    info = dict_map.get(var)
    if not info:
        return None

    label = clean_label(info["label"])
    vtype = info["type"]
    codes = info["codes"]

    value_str = str(int(value)) if isinstance(value, float) and value.is_integer() else str(value)

    # Skip No, Unknown, Missing unless you want them included
    if value_str in ["8", "9","99", "999"]:
        return None
    # Special free-text fields
    if var in ["sex", "age"]:
      return None
    if var in ["vtelldiedtx", "vknowcodtx"]:
        text = str(value).strip()
        if text == "":
            return None
        return f"{label}: {text}."
    else:
      if vtype == "categorical":
          decoded = codes.get(value_str, value_str)
          if decoded.lower() == "yes":
              return f"The deceased had {label.lower()}."
          else:
              return f"{label} was {decoded.lower()}."
      elif vtype == "numeric":
          return f"{label}: {value}."
    return None


def create_narrative(row):
    sentences = []
    demo = create_demographic_intro(row)
    if demo:
      sentences.append(demo)
    for var in dict_map.keys():
      # Stop reading variables at narr_avail
      if var == "narr_avail":
        break
      if var in row:
        sentence = describe_value(var, row[var])
        if sentence:
          sentences.append(sentence)

    return " ".join(sentences)

In [ ]:
def create_missing_narrative(row):

    missing_items = []

    for var in dict_map.keys():

        # Stop at narr_avail
        if var == "narr_avail":
            break

        if var not in row:
            continue

        value = row[var]

        info = dict_map.get(var)
        if not info:
            continue

        label = clean_label(info["label"])

        # Missing values
        if pd.isna(value):
            missing_items.append(label)
            continue

        value_str = str(value).strip().lower()

        # Empty values
        if value_str in ["", "na", "nan", "none"]:
            missing_items.append(label)
            continue

        # Unknown / Missing codes
        if value_str in ["8", "9", "99", "999"]:
            missing_items.append(label)

    if len(missing_items) == 0:
        return ""

    return (
        "The following information was not recorded or is unknown: "
        + ", ".join(missing_items)
        + "."
    )

In [ ]:
dictionary["code_map"] = dictionary["value_labels"].apply(parse_codes)

dict_map = {
    row["variable"]: {
        "label": row["variable_desc"],
        "type": row["data_type"],
        "codes": row["code_map"]
    }
    for _, row in dictionary.iterrows()
}

# Create narratives
data["generated_narrative"] = data.apply(create_narrative, axis=1)


In [ ]:
# Missing / unknown narrative
data["missing_narrative"] = data.apply(create_missing_narrative, axis=1)

In [ ]:
# Keep only ID and narrative
output = data[[
    "ident",
    "narrative",
    "generated_narrative",
    "missing_narrative"
]]
# Save output
output.to_excel("va_narratives.xlsx", index=False)